# Lab 3 — a genetic algorithm for the children's circle

Each child has $C$ boolean characteristics. The difference between two children is their Hamming distance (the number of characteristics on which they disagree). The goal is to **maximize the sum of differences along a circular seating order**, including the edge from the last child back to the first.

A genetic algorithm (GA) searches with a *population* of circles: evaluate them, select promising parents, recombine valid orders, mutate some offspring, and carry the best circles into the next generation. The notebook compares it with an exact search for small $N$ and a hill climber on the **same instance**. Neither heuristic proves optimality.

## 1. A reproducible instance and an evaluation budget

`INSTANCE_SEED` fixes the children across trials, so changes in the results come from the search rather than a different problem. The final experiment cell generates and prints a fresh trial seed on each run; both GA variants and the hill climber receive independent random generators initialized with that same number. Child indices are identities: two children may have identical characteristics but must still both appear in the circle. GA counts distinct complete-circle fitness evaluations; the hill climber counts score checks (one full score per restart, then one cheap matrix-based delta per proposed move).

In [1]:
from random import Random, SystemRandom
from time import perf_counter

from circle_optimization import (
    circle_fitness, distance_matrix, exact_best,
    format_distance_matrix, initial_population, mutate, order_crossover, random_individual,
    run_ga, run_hill_climber, tournament_selection, valid_circle,
)

NUM_CHILDREN = 40
NUM_CHARACTERISTICS = 100
INSTANCE_SEED = 2026
TRIAL_SEED_OVERRIDE: int | None = None  # Set to a printed trial seed to replay it
EXACT_LIMIT = 10
POPULATION_SIZE = 50
GENERATIONS = 200
TOURNAMENT_SIZE = 10
ELITES = 2
CROSSOVER_RATE = 0.9
BASELINE_MUTATION_RATE = 0.25
MUTATION_RATE = 0.5
EVALUATION_BUDGET = 10_000

instance_rng = Random(INSTANCE_SEED)
profiles = tuple(
    tuple(bool(instance_rng.getrandbits(1)) for _ in range(NUM_CHARACTERISTICS))
    for _ in range(NUM_CHILDREN)
)
for index, characteristics in enumerate(profiles):
    print(f"Child{index}: {''.join('+' if value else '-' for value in characteristics)}")

Child0: --++++++-+-++++-+++++++-+++---+---+---+++-+-----+++-+-++++-+--++----+++++--++------+++-+--++++++--++
Child1: -+-+++-+++-+-+-+-+++++-++----+-++-++-+-++++++-+-++-+++++-++----+----++++-+++--+--+--+++-+-+++-+---++
Child2: --+++++--++-+-++-+-+-+----++---+----++++-++---+-+++-+++----+++++++-+++-+--+-+++++---+-+-++++-+--+-++
Child3: ---+--+--+-+-+++--+--+---+--+-+++--++--+--+-++-+---++--++++--+++--+++--++-+++++--++----+-+---------+
Child4: ++-----+---++++-+-+---+-+-+--++++++-+--+--+++++--++-++++-+-+-++--+--+---+-+-++-++-+--++--+-------+++
Child5: +--+--------++++++--+-+++++-+--+-+--+++-+-+++++---++--++-+--+--+++++--+--++-++--+-++--+----+-++++---
Child6: +------+-+-+--+-++++++-+-+-----+---+--+++-+-+-++++++--++++--++-++---+-+++-++++-++-++-++--+----+-++-+
Child7: +++---+---+-+-+++++-+-+----++--++-++-+-++++-+-+--+++++++-++---+--+---++++--+--+---+--+++-+++-+--++++
Child8: ++++-++---+---+-++++++-+-----+-++---+---+++---+-+---+--++-+-++-++---++-+-+--++----++--+++++-----+--+
Child9: ++-++--+++-

## 2. Represent and score a circle

**From profiles to distances.** `profiles[i]` contains the Boolean characteristics of the child at index `i`. `distance_matrix(profiles)` compares every pair of profiles, counting positions where their characteristics differ (Hamming distance). The result is a lookup table: `distances[i][j]` is the difference between children `i` and `j`. Each child keeps its own index even if two profiles are identical; the diagonal is zero, and the table is symmetric. Below are the first ten rows and columns of the $40 \times 40$ table used by the GA.

**From distances to fitness.** A genome lists every child's index once, specifying their seating order. `circle_fitness` looks up and adds the distances between neighbors, including the last child back to the first: for the genome $(a, b, c)$, the score is $D_{ab} + D_{bc} + D_{ca}$, where $D_{ij}$ is `distances[i][j]`. Larger is better. The table stays fixed while the GA tries different orders: computing it once takes $O(N^2 C)$ time for $N$ children with $C$ characteristics, then each circle takes only $O(N)$ lookups to score.

Fixing child `0` removes rotated copies. Reversing a circle gives the same neighbors, so the lexicographically smaller direction is kept (for example, `(0, 3, 1, 2)` becomes `(0, 2, 1, 3)`). The resulting tuple is also a hashable key for detecting duplicate circles.

In [2]:
distances = distance_matrix(profiles)
print(format_distance_matrix(distances, max_children=10))

Distance matrix (40 × 40) — first 10 rows and columns:
       | Child0  Child1  Child2  Child3  Child4  Child5  Child6  Child7  Child8  Child9       …
-----------------------------------------------------------------------------------------------
Child0 |      0      42      47      52      51      56      44      48      52      53       …
Child1 |     42       0      51      52      55      60      48      44      48      61       …
Child2 |     47      51       0      55      52      53      51      47      41      44       …
Child3 |     52      52      55       0      45      54      42      50      46      47       …
Child4 |     51      55      52      45       0      47      43      43      55      52       …
Child5 |     56      60      53      54      47       0      42      52      52      43       …
Child6 |     44      48      51      42      43      42       0      48      40      57       …
Child7 |     48      44      47      50      43      52      48       0      46  

## 3. Build a population and its genetic operators

**Tournament selection:** draw $k$ = `TOURNAMENT_SIZE` population members without replacement and select the one with the highest fitness. A larger $k$ favors current winners more strongly but can reduce diversity.

**Order crossover (OX):** copy a slice of parent A's tail to the same positions in the child; scan parent B's tail cyclically after the slice and fill the remaining positions, skipping already-used children. Ordinary one-point crossover could duplicate children or leave some out. The anchored `0` never participates.

**Mutation:** either swap two tail positions or reverse a tail segment. Normalize afterward. These moves may occasionally reproduce an existing circle; the population builder rejects duplicate canonical genomes.

In [3]:
operator_rng = Random(99)
parent_a = tuple(range(NUM_CHILDREN))
parent_b = random_individual(NUM_CHILDREN, Random(97))
for _ in range(100):
    offspring = order_crossover(parent_a, parent_b, operator_rng)
    assert valid_circle(offspring, NUM_CHILDREN)
    assert valid_circle(mutate(offspring, operator_rng), NUM_CHILDREN)
assert len(set(initial_population(NUM_CHILDREN, POPULATION_SIZE, Random(99)))) == POPULATION_SIZE
contestants = initial_population(NUM_CHILDREN, TOURNAMENT_SIZE, Random(13))
score_contestant = lambda circle: circle_fitness(circle, distances)
assert score_contestant(tournament_selection(contestants, score_contestant, TOURNAMENT_SIZE, Random(5))) == max(map(score_contestant, contestants))
print('Example parents:', parent_a, parent_b)
print('Example OX child:', order_crossover(parent_a, parent_b, Random(7)))
print('OX and mutation preserve every child exactly once.')

Example parents: (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39) (0, 13, 28, 24, 4, 2, 3, 35, 18, 12, 1, 34, 21, 15, 16, 22, 36, 11, 8, 10, 6, 9, 14, 7, 20, 32, 33, 37, 38, 19, 29, 27, 31, 30, 17, 5, 25, 39, 26, 23)
Example OX child: (0, 2, 4, 24, 28, 23, 26, 39, 25, 5, 30, 31, 27, 29, 38, 37, 33, 32, 7, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 6, 8, 36, 22, 34, 1, 35, 3)
OX and mutation preserve every child exactly once.


## 4. Evolve generations

The two best individuals survive unchanged (**elitism**). Other places are filled by tournament-selected parents, crossover with probability `0.9`, and mutation with the rate passed to `run_ga`. A baseline rate of `0.25` is compared with `0.50`; this probability applies **once per offspring**, not independently to each position. A duplicate canonical circle is retried; after many collisions a fresh random circle is injected to maintain diversity.

Fitness values are cached by canonical genome. `evaluations` counts *new complete-circle scores*, not parent selections or cache hits. The budget is checked before building a generation, so even if all its members are new, they cannot exceed the common evaluation cap. The history records generation, best-so-far score, mean population score, mean number of seating positions differing from that generation's winner (a diversity measure), and cumulative evaluations.

`run_ga` and its documented genetic operators live in [`circle_optimization/classic/genetic.py`](circle_optimization/classic/genetic.py). The notebook keeps the experiment settings and displays the search history.

## 5. Compare searches on the same children

Rerun the **next code cell** for a fresh trial seed. The two GA rates (`0.25` and `0.50`) and the hill climber each get an independent `Random(trial_seed)` on the same children. The GA variants begin with the same random initial population, then diverge as their mutation decisions change the random-draw sequence. Equal numeric seeds make the trial reproducible, but do not make GA and hill climbing take identical moves. Set `TRIAL_SEED_OVERRIDE` to a printed seed to replay a trial.

For $N \le 10$, exhaustive search provides a proven optimum; above that limit it is skipped because the number of circles is $(N-1)!/2$. The hill climber proposes the same swap/reversal moves as the GA. Its numerical score-check limit is the **0.50-rate GA's actual number of distinct fitness evaluations**, but a HC delta check is cheaper than a GA complete-circle evaluation: equal numerical limits are not equal compute. The two GA rates can also have different actual evaluation counts. Compare score and elapsed search time (excluding the shared distance-matrix construction), not just these counts.

In [ ]:
def ga_trial(mutation_rate: float, seed: int) -> dict:
    """Run the shared GA on this fixed instance with a reproducible trial seed."""
    return run_ga(
        distances, Random(seed), population_size=POPULATION_SIZE,
        generations=GENERATIONS, tournament_size=TOURNAMENT_SIZE,
        elites=ELITES, crossover_rate=CROSSOVER_RATE,
        mutation_rate=mutation_rate, evaluation_budget=EVALUATION_BUDGET,
    )

trial_seed = (SystemRandom().getrandbits(64)
              if TRIAL_SEED_OVERRIDE is None else TRIAL_SEED_OVERRIDE)
print('Instance seed:', INSTANCE_SEED, '| Trial seed:', trial_seed)
start = perf_counter()
baseline_ga = ga_trial(BASELINE_MUTATION_RATE, trial_seed)
baseline_seconds = perf_counter() - start
start = perf_counter()
ga = ga_trial(MUTATION_RATE, trial_seed)
ga_seconds = perf_counter() - start
start = perf_counter()
hill = run_hill_climber(
    distances, Random(trial_seed), evaluation_budget=ga['evaluations'],
)
hill_seconds = perf_counter() - start
exact = exact_best(distances) if NUM_CHILDREN <= EXACT_LIMIT else None
print(f'generation | best | mean | positional diversity | evaluations (GA p={MUTATION_RATE:.2f})')
for generation, best, average, diversity, evaluations in ga['history']:
    if generation % 25 == 0 or generation == ga['history'][-1][0]:
        print(f'{generation:>10} | {best:>4} | {average:>4.1f} | {diversity:>20.1f} | {evaluations:>11}')
print(f"GA p={BASELINE_MUTATION_RATE:.2f}: score {baseline_ga['score']}, {baseline_ga['evaluations']} distinct evaluations, {baseline_seconds:.3f}s")
print(f"GA p={MUTATION_RATE:.2f}: score {ga['score']}, {ga['evaluations']} distinct evaluations, {ga_seconds:.3f}s")
print(f"Hill:      score {hill['score']}, {hill['evaluations']} score checks, {hill_seconds:.3f}s")
if exact is not None:
    print(f'Exact optimum: {exact[1]} (GA p={BASELINE_MUTATION_RATE:.2f} gap {exact[1] - baseline_ga["score"]}, GA p={MUTATION_RATE:.2f} gap {exact[1] - ga["score"]}, hill gap {exact[1] - hill["score"]})')
else:
    print('Exact search skipped at this N')
print(f'GA p={BASELINE_MUTATION_RATE:.2f} circle:', ' -> '.join(f'Child{i}' for i in (*baseline_ga['circle'], 0)))
print(f'GA p={MUTATION_RATE:.2f} circle:', ' -> '.join(f'Child{i}' for i in (*ga['circle'], 0)))
print('Hill circle:', ' -> '.join(f'Child{i}' for i in (*hill['circle'], 0)))

## Reading the result

`best` in the generation table never decreases because elites survive unchanged. `mean` and positional diversity show how quickly the population converges; duplicate prevention keeps every *current* population member distinct, but past circles may reappear and reuse cached scores. Compare the two mutation rates over several printed trial seeds on the same fixed instance—one trial cannot establish which is better. The hill climber's *numerical* score-check cap follows the 0.50-rate GA's distinct-evaluation count, but the counts measure different amounts of work. For larger $N$, exact search is skipped while the heuristic comparison still runs.